In [ ]:
Content:
1. LLD Approach
2.1 Parking Lot
2.2 Splitwise

# 1. LLD Approach

In [ ]:
LLD Approach:
    1. Clarify Requirements
       - Functional (what system should do)
       - Non-functional (scale, performance, concurrency)
    
    2. Identify Core Entities
       - Nouns → Classes
    
    3. Define Relationships
       - Association / Composition / Inheritance
    
    4. Define Interfaces / Contracts
       - Abstract classes / interfaces
    
    5. Design Class Diagram (mentally or text)
    
    6. Write Code (clean + modular)
    
    7. Add Extensibility
       - Strategy / Factory / Observer etc.
    
    8. Handle Edge Cases
    
    9. Discuss Trade-offs

In [ ]:
Must Have Patterns:
    Strategy: Payment / Pricing (Weekday, Weekend, Surge)
    Factory:  Object creation
    Singleton: Config / Logger
    Observer: Notifications
    Decorator: Add features dynamically


# 2.1 Parking Lot

In [ ]:
Client
  ↓
ParkingLotService  (Facade)
  ├── parkVehicle()
  ├── unparkVehicle()
  ├── getAvailableSlots()

Domain:
  ParkingLot → Floor → ParkingSlot
  Vehicle (Car, Bike)
  Ticket

Support:
  IParkingStrategy
  PaymentService

In [ ]:
public class ParkingLot
{
    public IReadOnlyCollection<Floor> Floors { get; } // Parking building has floors

    public ParkingLot(List<Floor> floors)
    {
        Floors = floors.AsReadOnly();
    }
}

public class Floor
{
    public int FloorNumber { get; }
    public IReadOnlyCollection<ParkingSlot> Slots { get; } // One Floor has multiple slots

    public Floor(int floorNumber, List<ParkingSlot> slots)
    {
        FloorNumber = floorNumber;
        Slots = slots.AsReadOnly();
    }
}

public class ParkingSlot
{
    private readonly object _lock = new object();

    public int SlotNumber { get; }
    public VehicleType SlotType { get; }
    public Vehicle ParkedVehicle { get; private set; }

    public bool IsAvailable => ParkedVehicle == null;

    public ParkingSlot(int slotNumber, VehicleType slotType)
    {
        SlotNumber = slotNumber;
        SlotType = slotType;
    }

    public bool TryPark(Vehicle vehicle)
    {
        lock (_lock)
        {
            if (!IsAvailable || vehicle.Type != SlotType)
                return false;

            ParkedVehicle = vehicle;
            return true;
        }
    }

    public void RemoveVehicle()
    {
        lock (_lock)
        {
            ParkedVehicle = null;
        }
    }
}



In [ ]:
// Nouns

public abstract class Vehicle
{
    public string Number { get; }
    public VehicleType Type { get; }

    protected Vehicle(string number, VehicleType type)
    {
        Number = number;
        Type = type;
    }
}

public class Car : Vehicle
{
    public Car(string number) : base(number, VehicleType.Car) { }
}

public class Bike : Vehicle
{
    public Bike(string number) : base(number, VehicleType.Bike) { }
}

public class Ticket
{
    public string TicketId { get; }
    public DateTime EntryTime { get; }
    public ParkingSlot Slot { get; }
    public Vehicle Vehicle { get; }

    public Ticket(string ticketId, ParkingSlot slot, Vehicle vehicle)
    {
        TicketId = ticketId;
        EntryTime = DateTime.UtcNow;
        Slot = slot;
        Vehicle = vehicle;
    }
}

In [ ]:
// Service for Injection

public interface IParkingStrategy
{
    ParkingSlot FindSlot(IReadOnlyCollection<Floor> floors, VehicleType type);
}

public class NearestSlotStrategy : IParkingStrategy
{
    public ParkingSlot FindSlot(IReadOnlyCollection<Floor> floors, VehicleType type)
    {
        foreach (var floor in floors)
        {
            foreach (var slot in floor.Slots)
            {
                if (slot.IsAvailable && slot.SlotType == type)
                    return slot;
            }
        }
        return null;
    }
}

public class PaymentService
{
    private const decimal RatePerHour = 10;

    public decimal CalculateAmount(Ticket ticket)
    {
        var hours = Math.Ceiling((DateTime.UtcNow - ticket.EntryTime).TotalHours);
        return (decimal)hours * RatePerHour;
    }

    public void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Payment of {amount} processed.");
    }
}

In [ ]:
// Main Service

public class ParkingLotService
{
    private readonly ParkingLot _parkingLot;     // Data Model Class
    private readonly IParkingStrategy _strategy; // Injected Service 1
    private readonly PaymentService _paymentService; // Injected Service 2

    public ParkingLotService(
        ParkingLot parkingLot,
        IParkingStrategy strategy,
        PaymentService paymentService)
    {
        _parkingLot = parkingLot;
        _strategy = strategy;
        _paymentService = paymentService;
    }

    // 🚗 PARK VEHICLE
    public Ticket ParkVehicle(Vehicle vehicle)
    {
        var slot = _strategy.FindSlot(_parkingLot.Floors, vehicle.Type);

        if (slot == null || !slot.TryPark(vehicle))
            throw new Exception("No available slot");

        return new Ticket(Guid.NewGuid().ToString(), slot, vehicle);
    }

    // 🚗 UNPARK VEHICLE
    public decimal UnparkVehicle(Ticket ticket)
    {
        var amount = _paymentService.CalculateAmount(ticket);
        _paymentService.ProcessPayment(amount);

        ticket.Slot.RemoveVehicle();

        return amount;
    }

    // 📊 GET AVAILABLE SLOTS
    public IReadOnlyCollection<ParkingSlot> GetAvailableSlots(VehicleType type)
    {
        return _parkingLot.Floors
            .SelectMany(f => f.Slots)
            .Where(s => s.IsAvailable && s.SlotType == type)
            .ToList()
            .AsReadOnly();
    }
}


// Main Method

class Program
{
    static void Main()
    {
        var slots = new List<ParkingSlot>
        {
            new ParkingSlot(1, VehicleType.Car),
            new ParkingSlot(2, VehicleType.Car),
            new ParkingSlot(3, VehicleType.Bike)
        };

        var floor1 = new Floor(1, slots);

        var parkingLot = new ParkingLot(new List<Floor> { floor1 });

        var service = new ParkingLotService(
            parkingLot,
            new NearestSlotStrategy(),
            new PaymentService()
        );

        var car = new Car("KA01AB1234");

        Console.WriteLine("Parking vehicle...");
        var ticket = service.ParkVehicle(car);

        Console.WriteLine($"Ticket Generated: {ticket.TicketId}");

        Thread.Sleep(2000); // simulate time

        Console.WriteLine("Unparking vehicle...");
        var amount = service.UnparkVehicle(ticket);

        Console.WriteLine($"Total Paid: {amount}");
    }
}

# 2.2 Splitwise

# 2. Web Scraping

In [ ]:
Shipments/day = 10,000

Assumptions:
- Each shipment checked multiple times/day
- Avg checks per shipment in 1 day = 24 (in every 1 hr)

Total API calls/day = 10K * 24 = 240,000
- 167 requests per min
- 3 requests per sec.

- Retry failures 20%, so 4 requests per second.

# Assumptions
    - Concurrency = 50 depends on CPU
    - Latency = 1 sec (scraping)
    - Throughput: 50/1 = 50 req/sec
    - 1 vCPU → ~50 concurrent async I/O tasks. This is concurrent requests, not requests per second.
    - 2 vCPU, 1 CPU is enough, But we have 2 servers.


1. Cron Job, fetch Next 85 messages from the DB and push them into the queue.
Cron (every 1 min)
      ↓
Fetch due AWBs (next_check_at <= now)
      ↓
Push to Queue
      ↓
Workers
      ↓
Update status + next_check_at (if undelivered: next_check_at = now + 4 hours) else next_check=Null

# Practical usecase:
    - use SemaphoreSlim(10), this will make max 10 API calls in parallel.
    - It can go upto 50 for one core, we are keeping 10 because of airline API constraints.
        
